### **Setup: Imports und Initialisierung**

In [ ]:
import sys
import os
import pandas as pd
import re

sys.path.append(os.path.abspath('..'))

from quinex_application.quinex_processing.mongos_store import MongoStore
from quinex_application.quinex_processing.quinex_client import QuinexClient
from quinex_application.quinex_processing.quinex_postprocessing import QuinexPostprocessing

In [ ]:
quinex = QuinexClient()
mongodb = MongoStore(quinex, database_name="quinex_db_query4", collection_name="papers_2010")
qpost = QuinexPostprocessing()

### **Adapt Specific Columns**

In [ ]:
df = pd.read_parquet("../data/visualisation_files/df_technologies_nan.parquet")

In [ ]:
df.columns

#### Adapt Quote

In [ ]:
df["open_access"].value_counts()

In [ ]:
df["license_information"].value_counts()

In [ ]:
mask = (
    (df["open_access"] == False) &
    (df["license_information"].isna())
)

df.loc[mask, "quote"] = pd.NA

In [ ]:
len(df[df['quote'].notna()]), len(df[df['quote'].isna()])

#### Adapt Source

In [ ]:
def map_source(value):
    text = "" if pd.isna(value) else str(value).strip()

    # Falls keine URL enthalten ist -> Abstract
    has_url = bool(re.search(r"https?://", text, flags=re.IGNORECASE))
    if not has_url:
        return "Abstract (Scopus/WOS)"

    # URL enthalten: Elsevier, wenn 'elsevier' vorkommt, sonst OpenAlex
    if "elsevier" in text.lower():
        return "Elsevier"
    return "OpenAlex"

df["source_url"] = df["source_url"].map(map_source)
df["source_url"].value_counts(dropna=False)

### **Drop and Rename Columns**

In [ ]:
df.columns

In [ ]:
df = df.drop(columns=[
    'entity_position', 'property_position', 'quantity_position', 'quote_position', 'method_position', 'temporal_scope_position', 'spatial_scope_position', 'qualifier_position', 'classification'
])

In [ ]:
df = df.rename(columns={
    'source_url': 'Source',
    'open_access': 'Open Access',
    'license_information': 'License Information',
    'publisher': 'Publisher',
    'author': 'Author',
    'publication_year': 'Publication Year',
    'author_countries': 'Author Origin',
    'entity': 'Entity',
    'property': 'Property',
    'quantity_text': 'Quantity',
    'quote': 'Quote',
    'value_normalized': 'Value Normalized',
    'unit_normalized': 'Unit Normalized',
    "unit_dimension": "Unit Dimension",
    "quantity_normalized": "Quantity Normalized",
    'spatial_scope': 'Spatial Scope',
    'temporal_scope': 'Temporal Scope',
    'method': 'Method',
    'qualifier': 'Qualifier',
    'references': 'Reference',
    'entity_mapping': 'Entity Mapping',
    'property_mapping': 'Property Mapping',
    'entity_base_mapping': 'Entity Base Mapping',
    'property_base_mapping': 'Property Base Mapping',
})

In [ ]:
df = df.sort_values(
    by=["Publication Year", "DOI"],
    ascending=[True, True]
).reset_index(drop=True)

In [ ]:
df.to_parquet("../data/quinex/df_quinex_juelichdata.parquet")

In [ ]:
df.drop(columns=['Entity Mapping', 'Entity Base Mapping', 'Property Mapping', 'Property Base Mapping'], inplace=True)
df.to_parquet("../streamlit/duckdb_data/df_quinex.parquet")

### **Define Number of Metadata**

In [ ]:
df = pd.read_parquet("../data/quinex/df_quinex_juelichdata.parquet")
df.columns

In [ ]:
metadata_cols = [
    "DOI",
    "Author Origin",
    "Publisher",
    "Publication Year",
    "Entity",
    "Property",
    "Quote",
    "Temporal Scope",
    "Spatial Scope",
    "Reference",
    "Method",
    "Qualifier",
]

missing_cols = [c for c in metadata_cols if c not in df.columns]
if missing_cols:
    print("Fehlende Spalten:", missing_cols)

cols_present = [c for c in metadata_cols if c in df.columns]

# Standard: nicht-leer = nicht-NaN
non_empty_mask = df[cols_present].notna()

# Für diese Textspalten zählen auch leere Strings (inkl. Whitespaces) als fehlend
empty_string_cols = ["Entity", "Property", "Quote", "Method", "Qualifier"]
for col in empty_string_cols:
    if col in cols_present:
        non_empty_mask[col] = non_empty_mask[col] & df[col].astype(str).str.strip().ne("")

summary = pd.DataFrame({
    "non_empty_count": non_empty_mask.sum(),
    "total_rows": len(df),
})
summary["non_empty_pct"] = (summary["non_empty_count"] / summary["total_rows"] * 100).round(2)
summary

### **Investigate Share of Paper Reproduction**

In [ ]:
df_path = "../data/quinex/df_quinex.parquet"
df_quinex = pd.read_parquet(df_path)

In [ ]:
df_quinex.head()[["DOI", "open_access", "entity", "property", "quote"]]

In [ ]:
df_quinex[df_quinex["DOI"] == "10.1016/j.biombioe.2010.01.007"][["DOI", "quote_position", "temporal_scope_position", "spatial_scope_position", "method_position", "qualifier_position", "reference_position"]].head()

In [ ]:
per_doi_coverage = qpost.summarize_text_coverage(
    df_quinex
    .assign(quote=pd.NA, quote_position=None)
)
display(per_doi_coverage)
per_doi_coverage["covered_text_pct"] = per_doi_coverage["covered_text_pct"].round().astype(int)
per_doi_coverage[per_doi_coverage["covered_text_pct"] != 0]["covered_text_pct"].hist(bins=30)
plt.show()

### **Investigate Length of Text Snippets**

In [ ]:
import ast

pos_cols = [
    "entity_position", "property_position", "quantity_position", "quote_position", "temporal_scope_position", "spatial_scope_position",
    "method_position", "qualifier_position", "reference_position"
 ]

def pos_len(x):
    if x is None:
        return pd.NA
    
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s.lower() == "nan":
            return pd.NA
        try:
            x = ast.literal_eval(s)  # z.B. "(1, 10)" -> (1, 10)
        except Exception:
            return pd.NA

    if isinstance(x, (tuple, list)):
        vals = x
    elif hasattr(x, "tolist"):  # z.B. numpy array
        vals = x.tolist()
        if not isinstance(vals, list):
            vals = [vals]
    else:
        try:
            if pd.isna(x):
                return pd.NA
        except Exception:
            pass
        return pd.NA

    if len(vals) != 2:
        return pd.NA

    try:
        return float(vals[1]) - float(vals[0])
    except Exception:
        return pd.NA

length_df = df_quinex[pos_cols].apply(lambda s: s.map(pos_len))
avg_len = length_df.mean(skipna=True).dropna()

if avg_len.empty:
    print("Keine gueltigen Positions-Tupel gefunden (z.B. (start, ende)).")
else:
    ax = avg_len.plot(kind="bar", color="steelblue", figsize=(8, 4))
    ax.set_ylabel("Durchschnittliche Laenge")
    ax.set_title("Mittlere Laenge pro Positionsspalte")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

print("Gueltige Werte pro Spalte:")
display(length_df.notna().sum())

In [ ]:
df_quinex["quote_position"].value_counts()